# Cross validation of quantile models

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

MODEL_FILES = {
    "GPR": "gpr",
    "GPQR (Independent)": "gpqr_independent",
    "GPQR (LMC)": "gpqr_lmc",
    "GPQR (Center-gap LMC)": "gpqr_cglmc",
}


def read_benchmarks(metric, target, index_col):
    frames = []
    for label, model_file in MODEL_FILES.items():
        frame = pd.read_csv(
            f"../../benchmarks/v1/{metric}.{model_file}.csv",
            index_col=index_col,
        )
        frames.append(
            (label, frame[frame["target"] == target].drop(columns=["target"]))
        )
    return frames


def plot_nll(data, target):
    plt.figure(figsize=(8, 4))
    for label, frame in data:
        plt.bar([label], [frame["negative_log_likelihood"].mean()])
    plt.ylabel("Mean negative log-likelihood")
    plt.title(f"{target}: posterior predictive NLL")
    plt.xticks(rotation=20)
    plt.show()


def plot_pinball(data, target):
    means = [
        (label, frame.groupby("quantile_level")["loss"].mean()) for label, frame in data
    ]
    bar_width = 0.8 / len(means)
    x = np.arange(len(means[0][1]))
    plt.figure(figsize=(8, 4))
    for i, (label, values) in enumerate(means):
        offset = (i - (len(means) - 1) / 2) * bar_width
        plt.bar(x + offset, values, width=bar_width, label=label)
    plt.xlabel("Quantile level")
    plt.ylabel("Mean pinball loss")
    plt.title(f"{target}: pinball loss")
    plt.xticks(x, means[0][1].index.astype(str), rotation=45)
    plt.legend()
    plt.show()

## H

In [ ]:
likelihood = read_benchmarks("likelihood", "H", "index")
pinball = read_benchmarks("pinball_loss", "H", ["index", "quantile_level"])

In [ ]:
plot_nll(likelihood, "H")
plot_pinball(pinball, "H")

## phi_1

In [ ]:
likelihood = read_benchmarks("likelihood", "phi_1", "index")
pinball = read_benchmarks("pinball_loss", "phi_1", ["index", "quantile_level"])

In [ ]:
plot_nll(likelihood, "phi_1")
plot_pinball(pinball, "phi_1")

## phi_3

In [ ]:
likelihood = read_benchmarks("likelihood", "phi_3", "index")
pinball = read_benchmarks("pinball_loss", "phi_3", ["index", "quantile_level"])

In [ ]:
plot_nll(likelihood, "phi_3")
plot_pinball(pinball, "phi_3")